In [5]:
import asyncio
import csv
from playwright.async_api import async_playwright

URL = "https://www.pricecharting.com/console/pokemon-promo"
OUTPUT = "../data/pokemon_promo_prices.csv"


async def scrape():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"
        )

        print("Loading page...")
        await page.goto(URL, wait_until="networkidle", timeout=60000)

        # Scroll to bottom repeatedly to trigger lazy-loaded rows
        print("Scrolling to load all cards...")
        prev_count = 0
        for _ in range(30):
            await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
            await asyncio.sleep(1.5)
            count = await page.locator("table#games_table tbody tr").count()
            if count == prev_count:
                break
            prev_count = count
            print(f"  Loaded {count} rows so far...")

        print(f"Total rows found: {prev_count}")

        # Parse table headers to find column indices
        headers = await page.locator("table#games_table thead th").all_inner_texts()
        headers = [h.strip() for h in headers]
        print(f"Columns: {headers}")

        # Fixed column indices based on actual page structure:
        # ['\xa0', 'Card', 'Ungraded', 'Grade 9', 'PSA 10', '']
        name_idx = 1
        ungraded_idx = 2
        grade9_idx = 3
        grade10_idx = 4
        bgs_idx = None  # not present on this page

        rows = await page.locator("table#games_table tbody tr").all()

        results = []
        for row in rows:
            cells = await row.locator("td").all_inner_texts()
            if not cells:
                continue

            def safe_get(idx):
                if idx is not None and idx < len(cells):
                    return cells[idx].strip()
                return ""

            # Grab image URL from the <img> tag in the row
            img = row.locator("td img").first
            img_src = ""
            if await img.count() > 0:
                img_src = await img.get_attribute("src") or ""
                if img_src.startswith("/"):
                    img_src = "https://www.pricecharting.com" + img_src

            results.append({
                "Card Name": safe_get(name_idx) if name_idx is not None else cells[0].strip(),
                "Ungraded Price": safe_get(ungraded_idx),
                "Grade 9 Price": safe_get(grade9_idx),
                "PSA 10 Price": safe_get(grade10_idx),
                "Image URL": img_src,
            })

        await browser.close()

        # Write CSV
        if results:
            with open(OUTPUT, "w", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=results[0].keys())
                writer.writeheader()
                writer.writerows(results)
            print(f"\nDone! Saved {len(results)} cards to '{OUTPUT}'")
        else:
            print("No data found — the page structure may have changed.")


if __name__ == "__main__":
    import sys
    if "ipykernel" in sys.modules:
        # Running in Jupyter — use the existing event loop
        import nest_asyncio
        nest_asyncio.apply()
        asyncio.get_event_loop().run_until_complete(scrape())
    else:
        asyncio.run(scrape())

Loading page...
Scrolling to load all cards...
  Loaded 100 rows so far...
  Loaded 150 rows so far...
  Loaded 200 rows so far...
  Loaded 250 rows so far...
  Loaded 300 rows so far...
  Loaded 350 rows so far...
  Loaded 400 rows so far...
  Loaded 450 rows so far...
  Loaded 500 rows so far...
  Loaded 550 rows so far...
  Loaded 600 rows so far...
  Loaded 650 rows so far...
  Loaded 700 rows so far...
  Loaded 750 rows so far...
  Loaded 800 rows so far...
  Loaded 850 rows so far...
  Loaded 900 rows so far...
  Loaded 950 rows so far...
  Loaded 1000 rows so far...
  Loaded 1050 rows so far...
  Loaded 1100 rows so far...
  Loaded 1150 rows so far...
  Loaded 1200 rows so far...
  Loaded 1250 rows so far...
  Loaded 1300 rows so far...
  Loaded 1350 rows so far...
  Loaded 1400 rows so far...
  Loaded 1450 rows so far...
  Loaded 1500 rows so far...
  Loaded 1550 rows so far...
Total rows found: 1550
Columns: ['', 'Card', 'Ungraded', 'Grade 9', 'PSA 10', '']

Done! Saved 1550 c

In [8]:
import pandas as pd
df = pd.read_csv("../data/pokemon_promo_prices.csv")

In [11]:
df.head()

,Card Name,Ungraded Price,Grade 9 Price,PSA 10 Price,Image URL
0,Mega Charizard X EX #23,$41.13,$59.99,$343.00,https://storage.googleapis.com/images.pricecha...
1,Pikachu with Grey Felt Hat #85,$846.46,$865.66,"$2,543.59",https://storage.googleapis.com/images.pricecha...
2,Ancient Mew,$78.74,$182.28,"$2,141.14",https://storage.googleapis.com/images.pricecha...
3,Oricorio EX #24,$11.56,$23.45,$134.38,https://storage.googleapis.com/images.pricecha...
4,Mew ex #53,$84.09,$108.75,$780.00,https://storage.googleapis.com/images.pricecha...


In [12]:
df.count()

Card Name         1550
Ungraded Price    1533
Grade 9 Price     1254
PSA 10 Price      1216
Image URL         1550
dtype: int64

In [13]:
import csv
import re
from collections import defaultdict

STRIP = re.compile(
    r'\b(EX|GX|V|VMAX|VSTAR|ex|Tag Team|Mega|MEGA|Break|BREAK|Radiant|'
    r'Prism Star|Star|Prime|Legend|SP|LV\.X|#\d+.*$)',
    re.IGNORECASE
)

def extract_species(name):
    species = STRIP.sub("", name)
    species = re.sub(r'\s+', ' ', species).strip().rstrip("-& ")
    return species

def parse_price(s):
    try:
        return float(s.replace("$", "").replace(",", "").strip())
    except:
        return 0.0

# Read existing CSV
with open("../data/pokemon_promo_prices.csv", newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# Add species and numeric ungraded price
for r in rows:
    r["Species"] = extract_species(r["Card Name"])
    r["_ungraded_num"] = parse_price(r["Ungraded Price"])

# Group by species, take max ungraded price per species
species_max = defaultdict(float)
for r in rows:
    species_max[r["Species"]] = max(species_max[r["Species"]], r["_ungraded_num"])

# Sort rows by species max price desc, then by card price desc
rows.sort(key=lambda r: (species_max[r["Species"]], r["_ungraded_num"]), reverse=True)

# Write sorted CSV
fieldnames = ["Species", "Card Name", "Ungraded Price", "Grade 9 Price", "PSA 10 Price", "Image URL"]
# Only include fieldnames that exist in the data
fieldnames = [f for f in fieldnames if f in rows[0]]

with open("../data/pokemon_promo_sorted.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(rows)

print(f"Saved sorted CSV to 'pokemon_promo_sorted.csv'")
print("\nTop 10 most expensive Pokémon species (by max ungraded price):")
seen = set()
count = 0
for r in rows:
    if r["Species"] not in seen:
        print(f"  {r['Species']}: {r['Ungraded Price']}")
        seen.add(r["Species"])
        count += 1
        if count == 10:
            break

Saved sorted CSV to 'pokemon_promo_sorted.csv'

Top 10 most expensive Pokémon species (by max ungraded price):
  Gengar [Prerelease Staff] #SWSH241: $2,000.00 
  Champions Festival [Top Thirty-Two] #XY27: $1,600.00 
  Tropical Wind #DP05: $1,594.69 
  Lucky Stadium #41: $1,407.54 
  Championship Arena #28: $1,339.87 
  Champions Festival [Quarter Finalist] #XY176: $1,170.00 
  Pokemon Center #40: $1,013.90 
  Champions Festival [Quarter Finalist] #SM231: $1,000.00 
  Tropical Tidal Wave [Worlds Staff] #36: $999.95 
  Champions Festival [Finalist] #XY176: $900.00 


In [14]:
with open("../data/pokemon_promo_prices.csv", newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

for r in rows[:10]:
    print(r["Card Name"])

Mega Charizard X EX #23
Pikachu with Grey Felt Hat #85
Ancient Mew
Oricorio EX #24
Mew ex #53
Charizard VStar #SWSH262
Moltres & Zapdos & Articuno GX #SM210
Charmander #44
Eevee #173
Charizard VMax #SWSH261


In [18]:
import csv
import re
from collections import defaultdict

STRIP = re.compile(r'\s*#\S+.*$|\b(EX|GX|VMAX|VSTAR|VMax|VStar|ex|Tag Team|Mega|Break|BREAK|Radiant|Prism Star|Prime|Legend|SP)\b', re.IGNORECASE)

def extract_species(name):
    species = re.sub(r'\[.*?\]', '', name)  # remove [Snowflake Stamp], [Winner] etc.
    species = STRIP.sub("", species)
    species = re.sub(r'\s+', ' ', species).strip().rstrip("-& ")
    return species

def parse_price(s):
    try:
        return float(s.replace("$", "").replace(",", "").strip())
    except:
        return 0.0

# Read existing CSV
with open("../data/pokemon_promo_prices.csv", newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# Add species and numeric ungraded price
for r in rows:
    r["Species"] = extract_species(r["Card Name"])
    r["_ungraded_num"] = parse_price(r["Ungraded Price"])

# Group by species, take max ungraded price per species
species_max = defaultdict(float)
for r in rows:
    species_max[r["Species"]] = max(species_max[r["Species"]], r["_ungraded_num"])

# Sort rows by species max price desc, then by card price desc
rows.sort(key=lambda r: (species_max[r["Species"]], r["_ungraded_num"]), reverse=True)

# Write sorted CSV
fieldnames = [f for f in ["Species", "Card Name", "Ungraded Price", "Grade 9 Price", "PSA 10 Price", "Image URL"] if f in rows[0]]

with open("../data/pokemon_promo_sorted.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(rows)

print("Saved to 'pokemon_promo_sorted.csv'")
print("\nTop 10 most expensive Pokemon species (by max ungraded price):")
seen = set()
count = 0
for r in rows:
    if r["Species"] not in seen:
        print(f"  {r['Species']}: {r['Ungraded Price']}")
        seen.add(r["Species"])
        count += 1
        if count == 10:
            break

Saved to 'pokemon_promo_sorted.csv'

Top 10 most expensive Pokemon species (by max ungraded price):
  Gengar: $2,000.00 
  Champions Festival: $1,600.00 
  Tropical Wind: $1,594.69 
  Lucky Stadium: $1,407.54 
  Championship Arena: $1,339.87 
  Pokemon Center: $1,013.90 
  Tropical Tidal Wave: $999.95 
  _____'s Mew: $850.00 
  Pikachu with Grey Felt Hat: $846.46 
  Tin: Pikachu & Zekrom: $793.67 


In [21]:
sorted = pd.read_csv("../data/pokemon_promo_sorted.csv")

In [22]:
sorted.head()

,Species,Card Name,Ungraded Price,Grade 9 Price,PSA 10 Price,Image URL
0,Gengar,Gengar [Prerelease Staff] #SWSH241,"$2,000.00","$3,218.83","$3,863.00",https://storage.googleapis.com/images.pricecha...
1,Gengar,Mega Gengar EX #XY166,$138.94,$616.10,"$6,156.60",https://storage.googleapis.com/images.pricecha...
2,Gengar,Gengar #SWSH052,$83.86,$149.44,"$1,970.12",https://storage.googleapis.com/images.pricecha...
3,Gengar,Gengar #SWSH241,$27.27,$46.20,$325.00,https://storage.googleapis.com/images.pricecha...
4,Champions Festival,Champions Festival [Top Thirty-Two] #XY27,"$1,600.00",$276.00,NaN,https://storage.googleapis.com/images.pricecha...


In [23]:
sorted.sort_values("Ungraded Price", ascending=False).head(10)

,Species,Card Name,Ungraded Price,Grade 9 Price,PSA 10 Price,Image URL
27,Tropical Tidal Wave,Tropical Tidal Wave [Worlds Staff] #36,$999.95,$804.15,"$3,734.85",https://storage.googleapis.com/images.pricecha...
404,Tropical Beach,Tropical Beach #BW28,$99.99,"$1,225.00","$2,608.19",https://storage.googleapis.com/images.pricecha...
406,Darkrai,Darkrai #BW73,$99.55,$651.63,"$7,915.11",https://storage.googleapis.com/images.pricecha...
414,Evolving Powers Premium Collection,Evolving Powers Premium Collection,$98.99,NaN,NaN,https://storage.googleapis.com/images.pricecha...
415,Comfey,Comfey [Prerelease Staff] #SWSH242,$98.57,$66.38,NaN,https://storage.googleapis.com/images.pricecha...
417,Charizard Super Premium Collection Box,Charizard Ex Super Premium Collection Box,$97.03,NaN,NaN,https://storage.googleapis.com/images.pricecha...
44,Pikachu,Pikachu [World Championships Winner] #225,$95.05,$233.76,NaN,https://storage.googleapis.com/images.pricecha...
418,Kingdra,Kingdra [Prerelease Staff] #XY39,$95.00,NaN,NaN,https://storage.googleapis.com/images.pricecha...
422,Grovyle,Grovyle #004,$94.95,$107.39,$212.36,https://storage.googleapis.com/images.pricecha...
423,Victory Medal,Victory Medal [2007-2008 Battle Road Autumn],$94.57,$97.56,"$3,500.00",https://storage.googleapis.com/images.pricecha...


In [24]:
import os
os.environ["R_HOME"] = "/Library/Frameworks/R.framework/Resources"

In [25]:
%reload_ext rpy2.ipython
%reload_ext autoreload
%autoreload 2

%matplotlib inline  
from matplotlib import rcParams
rcParams['figure.figsize'] = (16, 100)

import warnings
from rpy2.rinterface import RRuntimeWarning
warnings.filterwarnings("ignore") # Ignore all warnings
# warnings.filterwarnings("ignore", category=RRuntimeWarning) # Show some warnings

In [26]:
%%R

require('tidyverse')

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


Loading required package: tidyverse


In [30]:
%%R

sorted <- read_csv("../data/pokemon_promo_sorted.csv")

Rows: 1550 Columns: 6
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (6): Species, Card Name, Ungraded Price, Grade 9 Price, PSA 10 Price, Im...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [ ]:
%%R

sorted <- sorted %>%
  mutate(`Ungraded Price` = as.numeric(gsub("[\\$,]", "", `Ungraded Price`)))

top <- sorted %>%
  group_by(Species) %>%
  summarise(`Ungraded Price` = max(`Ungraded Price`, na.rm = TRUE)) %>%
  arrange(desc(`Ungraded Price`)) %>%
  head(15)

ggplot(top, aes(x = reorder(Species, `Ungraded Price`), y = `Ungraded Price`)) +
    geom_bar(stat = "identity", fill = "steelblue") +
    coord_flip() +
    labs(title = "Top 15 Most Expensive Pokémon Promo Cards by Species",
         x = "Species",
         y = "Ungraded Price (USD)")

In [ ]:
import asyncio
import csv
from playwright.async_api import async_playwright

URL_151 = "https://www.pricecharting.com/console/pokemon-chinese-151-collect"
OUTPUT_151 = "../data/pokemon_chinese_151_prices.csv"


async def scrape_151():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"
        )

        print("Loading page...")
        await page.goto(URL_151, wait_until="networkidle", timeout=60000)

        # Scroll to bottom repeatedly to trigger lazy-loaded rows
        print("Scrolling to load all cards...")
        prev_count = 0
        for _ in range(30):
            await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
            await asyncio.sleep(1.5)
            count = await page.locator("table#games_table tbody tr").count()
            if count == prev_count:
                break
            prev_count = count
            print(f"  Loaded {count} rows so far...")

        print(f"Total rows found: {prev_count}")

        # Parse table headers to find column indices
        headers = await page.locator("table#games_table thead th").all_inner_texts()
        headers = [h.strip() for h in headers]
        print(f"Columns: {headers}")

        name_idx = 1
        ungraded_idx = 2
        grade9_idx = 3
        grade10_idx = 4

        rows = await page.locator("table#games_table tbody tr").all()

        results = []
        for row in rows:
            cells = await row.locator("td").all_inner_texts()
            if not cells:
                continue

            def safe_get(idx):
                if idx is not None and idx < len(cells):
                    return cells[idx].strip()
                return ""

            # Grab image URL from the <img> tag in the row
            img = row.locator("td img").first
            img_src = ""
            if await img.count() > 0:
                img_src = await img.get_attribute("src") or ""
                if img_src.startswith("/"):
                    img_src = "https://www.pricecharting.com" + img_src

            results.append({
                "Card Name": safe_get(name_idx) if name_idx is not None else cells[0].strip(),
                "Ungraded Price": safe_get(ungraded_idx),
                "Grade 9 Price": safe_get(grade9_idx),
                "PSA 10 Price": safe_get(grade10_idx),
                "Image URL": img_src,
            })

        await browser.close()

        # Write CSV
        if results:
            with open(OUTPUT_151, "w", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=results[0].keys())
                writer.writeheader()
                writer.writerows(results)
            print(f"\nDone! Saved {len(results)} cards to '{OUTPUT_151}'")
        else:
            print("No data found — the page structure may have changed.")


import sys
if "ipykernel" in sys.modules:
    import nest_asyncio
    nest_asyncio.apply()
    asyncio.get_event_loop().run_until_complete(scrape_151())
else:
    asyncio.run(scrape_151())

In [ ]:
import csv
import re
from collections import defaultdict

STRIP = re.compile(r'\s*#\S+.*$|\b(EX|GX|VMAX|VSTAR|VMax|VStar|ex|Tag Team|Mega|Break|BREAK|Radiant|Prism Star|Prime|Legend|SP)\b', re.IGNORECASE)

def extract_species(name):
    species = re.sub(r'\[.*?\]', '', name)  # remove [Snowflake Stamp], [Winner] etc.
    species = STRIP.sub("", species)
    species = re.sub(r'\s+', ' ', species).strip().rstrip("-& ")
    return species

def parse_price(s):
    try:
        return float(s.replace("$", "").replace(",", "").strip())
    except:
        return 0.0

# Read existing CSV
with open("../data/pokemon_chinese_151_prices.csv", newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# Add species and numeric ungraded price
for r in rows:
    r["Species"] = extract_species(r["Card Name"])
    r["_ungraded_num"] = parse_price(r["Ungraded Price"])

# Group by species, take max ungraded price per species
species_max = defaultdict(float)
for r in rows:
    species_max[r["Species"]] = max(species_max[r["Species"]], r["_ungraded_num"])

# Sort rows by species max price desc, then by card price desc
rows.sort(key=lambda r: (species_max[r["Species"]], r["_ungraded_num"]), reverse=True)

# Write sorted CSV
fieldnames = [f for f in ["Species", "Card Name", "Ungraded Price", "Grade 9 Price", "PSA 10 Price", "Image URL"] if f in rows[0]]

with open("../data/pokemon_chinese_151_sorted.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(rows)

print(f"Saved sorted CSV to 'pokemon_chinese_151_sorted.csv'")
print(f"Total cards: {len(rows)}")
print("\nTop 10 most expensive Pokémon species (by max ungraded price):")
seen = set()
count = 0
for r in rows:
    if r["Species"] not in seen:
        print(f"  {r['Species']}: {r['Ungraded Price']}")
        seen.add(r["Species"])
        count += 1
        if count == 10:
            break

In [ ]:
import pandas as pd
df_151 = pd.read_csv("../data/pokemon_chinese_151_sorted.csv")
df_151.head(10)

In [ ]:
import asyncio
import csv
import re
from collections import defaultdict
from playwright.async_api import async_playwright

# 8 most expensive Pokemon sets (excluding Promo and Chinese 151)
SETS = [
    ("pokemon-base-set", "../data/pokemon_base_set"),
    ("pokemon-skyridge", "../data/pokemon_skyridge"),
    ("pokemon-aquapolis", "../data/pokemon_aquapolis"),
    ("pokemon-neo-destiny", "../data/pokemon_neo_destiny"),
    ("pokemon-legendary-collection", "../data/pokemon_legendary_collection"),
    ("pokemon-expedition", "../data/pokemon_expedition"),
    ("pokemon-deoxys", "../data/pokemon_deoxys"),
    ("pokemon-dragon-frontiers", "../data/pokemon_dragon_frontiers"),
]

STRIP = re.compile(
    r'\s*#\S+.*$|\b(EX|GX|VMAX|VSTAR|VMax|VStar|ex|Tag Team|Mega|Break|BREAK|Radiant|Prism Star|Prime|Legend|SP)\b',
    re.IGNORECASE,
)


def extract_species(name):
    species = re.sub(r'\[.*?\]', '', name)
    species = STRIP.sub("", species)
    species = re.sub(r'\s+', ' ', species).strip().rstrip("-& ")
    return species


def parse_price(s):
    try:
        return float(s.replace("$", "").replace(",", "").strip())
    except:
        return 0.0


async def scrape_set(slug, file_prefix):
    url = f"https://www.pricecharting.com/console/{slug}"
    raw_csv = f"{file_prefix}_prices.csv"
    sorted_csv = f"{file_prefix}_sorted.csv"

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"
        )

        print(f"\n{'='*60}")
        print(f"Scraping: {slug}")
        print(f"{'='*60}")
        await page.goto(url, wait_until="networkidle", timeout=60000)

        # Scroll to load all rows
        prev_count = 0
        for _ in range(30):
            await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
            await asyncio.sleep(1.5)
            count = await page.locator("table#games_table tbody tr").count()
            if count == prev_count:
                break
            prev_count = count
            print(f"  Loaded {count} rows...")

        print(f"  Total rows: {prev_count}")

        headers = await page.locator("table#games_table thead th").all_inner_texts()
        headers = [h.strip() for h in headers]
        print(f"  Columns: {headers}")

        name_idx = 1
        ungraded_idx = 2
        grade9_idx = 3
        grade10_idx = 4

        rows = await page.locator("table#games_table tbody tr").all()

        results = []
        for row in rows:
            cells = await row.locator("td").all_inner_texts()
            if not cells:
                continue

            def safe_get(idx):
                if idx is not None and idx < len(cells):
                    return cells[idx].strip()
                return ""

            img = row.locator("td img").first
            img_src = ""
            if await img.count() > 0:
                img_src = await img.get_attribute("src") or ""
                if img_src.startswith("/"):
                    img_src = "https://www.pricecharting.com" + img_src

            results.append({
                "Card Name": safe_get(name_idx),
                "Ungraded Price": safe_get(ungraded_idx),
                "Grade 9 Price": safe_get(grade9_idx),
                "PSA 10 Price": safe_get(grade10_idx),
                "Image URL": img_src,
            })

        await browser.close()

    if not results:
        print(f"  No data found for {slug}!")
        return

    # Write raw CSV
    with open(raw_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=results[0].keys())
        writer.writeheader()
        writer.writerows(results)
    print(f"  Saved {len(results)} cards to '{raw_csv}'")

    # Sort by species
    for r in results:
        r["Species"] = extract_species(r["Card Name"])
        r["_ungraded_num"] = parse_price(r["Ungraded Price"])

    species_max = defaultdict(float)
    for r in results:
        species_max[r["Species"]] = max(species_max[r["Species"]], r["_ungraded_num"])

    results.sort(key=lambda r: (species_max[r["Species"]], r["_ungraded_num"]), reverse=True)

    fieldnames = ["Species", "Card Name", "Ungraded Price", "Grade 9 Price", "PSA 10 Price", "Image URL"]
    with open(sorted_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(results)
    print(f"  Saved sorted CSV to '{sorted_csv}'")

    # Show top 5
    seen = set()
    count = 0
    print(f"  Top 5 species by ungraded price:")
    for r in results:
        if r["Species"] not in seen:
            print(f"    {r['Species']}: {r['Ungraded Price']}")
            seen.add(r["Species"])
            count += 1
            if count == 5:
                break


async def scrape_all():
    for slug, file_prefix in SETS:
        await scrape_set(slug, file_prefix)
    print(f"\n{'='*60}")
    print("All done! Scraped 8 sets.")
    print(f"{'='*60}")


import sys
if "ipykernel" in sys.modules:
    import nest_asyncio
    nest_asyncio.apply()
    asyncio.get_event_loop().run_until_complete(scrape_all())
else:
    asyncio.run(scrape_all())

In [5]:
import pandas as pd
import glob

sorted_files = [
    ("../data/pokemon_promo_sorted.csv", "Promo"),
    ("../data/pokemon_chinese_151_sorted.csv", "Chinese 151"),
    ("../data/pokemon_base_set_sorted.csv", "Base Set"),
    ("../data/pokemon_skyridge_sorted.csv", "Skyridge"),
    ("../data/pokemon_aquapolis_sorted.csv", "Aquapolis"),
    ("../data/pokemon_neo_destiny_sorted.csv", "Neo Destiny"),
    ("../data/pokemon_legendary_collection_sorted.csv", "Legendary Collection"),
    ("../data/pokemon_expedition_sorted.csv", "Expedition"),
    ("../data/pokemon_deoxys_sorted.csv", "EX Deoxys"),
    ("../data/pokemon_dragon_frontiers_sorted.csv", "EX Dragon Frontiers"),
]

dfs = []
for filename, set_name in sorted_files:
    try:
        df = pd.read_csv(filename)
        df["Set"] = set_name
        dfs.append(df)
        print(f"Loaded {filename}: {len(df)} cards")
    except FileNotFoundError:
        print(f"WARNING: {filename} not found, skipping")

merged = pd.concat(dfs, ignore_index=True)

# Reorder columns so Set comes first
cols = ["Set"] + [c for c in merged.columns if c != "Set"]
merged = merged[cols]

merged.to_csv("../data/pokemon_all_sorted.csv", index=False)
print(f"\nMerged {len(merged)} total cards from {len(dfs)} sets into 'pokemon_all_sorted.csv'")
merged.head(10)

Loaded pokemon_promo_sorted.csv: 1550 cards
Loaded pokemon_chinese_151_sorted.csv: 481 cards
Loaded pokemon_base_set_sorted.csv: 462 cards
Loaded pokemon_skyridge_sorted.csv: 339 cards
Loaded pokemon_aquapolis_sorted.csv: 344 cards
Loaded pokemon_neo_destiny_sorted.csv: 232 cards
Loaded pokemon_legendary_collection_sorted.csv: 234 cards
Loaded pokemon_expedition_sorted.csv: 340 cards
Loaded pokemon_deoxys_sorted.csv: 213 cards
Loaded pokemon_dragon_frontiers_sorted.csv: 200 cards

Merged 4395 total cards from 10 sets into 'pokemon_all_sorted.csv'


,Set,Species,Card Name,Ungraded Price,Grade 9 Price,PSA 10 Price,Image URL
0,Promo,Gengar,Gengar [Prerelease Staff] #SWSH241,"$2,000.00","$3,218.83","$3,863.00",https://storage.googleapis.com/images.pricecha...
1,Promo,Gengar,Mega Gengar EX #XY166,$138.94,$616.10,"$6,156.60",https://storage.googleapis.com/images.pricecha...
2,Promo,Gengar,Gengar #SWSH052,$83.86,$149.44,"$1,970.12",https://storage.googleapis.com/images.pricecha...
3,Promo,Gengar,Gengar #SWSH241,$27.27,$46.20,$325.00,https://storage.googleapis.com/images.pricecha...
4,Promo,Champions Festival,Champions Festival [Top Thirty-Two] #XY27,"$1,600.00",$276.00,NaN,https://storage.googleapis.com/images.pricecha...
5,Promo,Champions Festival,Champions Festival [Quarter Finalist] #XY176,"$1,170.00","$1,700.00",NaN,https://storage.googleapis.com/images.pricecha...
6,Promo,Champions Festival,Champions Festival [Quarter Finalist] #SM231,"$1,000.00","$2,968.84",NaN,https://storage.googleapis.com/images.pricecha...
7,Promo,Champions Festival,Champions Festival [Finalist] #XY176,$900.00,NaN,NaN,https://storage.googleapis.com/images.pricecha...
8,Promo,Champions Festival,Champions Festival [Staff] #XY176,$515.65,"$2,075.00",NaN,https://storage.googleapis.com/images.pricecha...
9,Promo,Champions Festival,Champions Festival [Staff] #SWSH296,$425.00,$382.74,$900.39,https://storage.googleapis.com/images.pricecha...
